# DQN with Prioritized Learning Replay

In [ ]:
import gymnasium as gym
import icu_sepsis

import numpy as np
import matplotlib
matplotlib.rcParams['agg.path.chunksize'] = 10000
import matplotlib.pyplot as plt
import os
import re
import glob
import random
from itertools import product

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from gymnasium.vector import SyncVectorEnv

env = gym.make('Sepsis/ICU-Sepsis-v2')

state, info = env.reset()
print('Initial state:', state)
print('Extra info:', info)

next_state, reward, terminated, truncated, info = env.step(0)
print('\nTaking action 0:')
print('Next state:', next_state)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)

print("Actions and State Space:")

print("Action space:", env.action_space)
print("State space:", env.observation_space)

In [ ]:
# Prioritized Replay Buffer to store transitions
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.pos = 0
        self.buffer = []
        self.priorities = []

    def add(self, s, a, r, s2, done):
        max_prio = max(self.priorities, default=1.0)
        if len(self.buffer) < self.capacity:
            self.buffer.append((s, a, r, s2, done))
            self.priorities.append(max_prio)
        else:
            self.buffer[self.pos] = (s, a, r, s2, done)
            self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        prios = np.array(self.priorities) ** self.alpha
        probs = prios / prios.sum()
        idxs = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[i] for i in idxs]
        total = len(self.buffer)
        weights = (total * probs[idxs]) ** (-beta)
        weights /= weights.max()
        return samples, idxs, weights

    def update_priorities(self, idxs, errors, eps=1e-6):
        for i, e in zip(idxs, errors):
            self.priorities[i] = abs(e) + eps


# Q-Network
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)
        self.apply(init_weights_he)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

def init_weights_he(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

# DQN Agent class
class DQNAgent:
    def __init__(self,
                 env_name,
                 lr=1e-3,
                 gamma=1.0,
                 buffer_size=10_000,
                 batch_size=64,
                 eps_start=1.0,
                 eps_end=0.001,
                 alpha=0.6,
                 beta=0.4,
                 learning_starts=10_000,
                 train_freq=10,
                 target_update=512,
                 device='cpu'):
        # environment
        self.env = gym.make(env_name)
        self.state_dim = self.env.observation_space.n
        self.action_dim = self.env.action_space.n
        self.device = torch.device(device)
        self.gamma = gamma

        # networks
        self.q_net = QNetwork(self.state_dim, self.action_dim).to(self.device)
        self.target_net = QNetwork(self.state_dim, self.action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.opt = optim.Adam(self.q_net.parameters(), lr=lr)

        # replay & hyperparams
        self.buffer = PrioritizedReplayBuffer(buffer_size, alpha)
        self.batch_size = batch_size
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.epsilon = eps_start
        self.beta = beta
        self.learning_starts = learning_starts
        self.train_freq = train_freq
        self.target_update = target_update

        # counters
        self.total_steps = 0
        self.learn_step = 0

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        s = F.one_hot(torch.tensor(state), self.state_dim).float().unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self.q_net(s).argmax().item()

    def train_one_episode(self, episode, max_episodes):
        # linear epsilon schedule over first 25% of episodes
        frac = min(1.0, episode / (0.25 * max_episodes))
        self.epsilon = self.eps_start + frac * (self.eps_end - self.eps_start)

        state, _ = self.env.reset()
        done = False
        total_reward, length, inad = 0.0, 0, 0

        while not done:
            length += 1
            action = self.select_action(state)
            next_state, reward, term, trunc, info = self.env.step(action)
            done = term or trunc
            total_reward += reward
            if info.get('inadmissible', False):
                inad += 1

            # store transition
            self.buffer.add(state, action, reward, next_state, done)
            state = next_state
            self.total_steps += 1

            # learning step
            if (self.total_steps >= self.learning_starts and
                self.total_steps % self.train_freq == 0 and
                len(self.buffer.buffer) >= self.batch_size):

                samples, idxs, weights = self.buffer.sample(self.batch_size, self.beta)
                states, actions, rewards, next_states, dones = zip(*samples)

                sb = F.one_hot(torch.tensor(states), self.state_dim).float().to(self.device)
                ab = torch.tensor(actions).unsqueeze(1).to(self.device)
                rb = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1).to(self.device)
                nsb = F.one_hot(torch.tensor(next_states), self.state_dim).float().to(self.device)
                db = torch.tensor(dones, dtype=torch.float32).unsqueeze(1).to(self.device)
                wb = torch.tensor(weights).unsqueeze(1).to(self.device)

                q_vals = self.q_net(sb).gather(1, ab)
                with torch.no_grad():
                    max_next = self.target_net(nsb).max(1, keepdim=True)[0]
                    target = rb + self.gamma * max_next * (1.0 - db)

                td_err = (target - q_vals).detach().cpu().numpy().flatten()
                loss = (wb * (q_vals - target).pow(2)).mean()

                self.opt.zero_grad()
                loss.backward()
                self.opt.step()
                self.buffer.update_priorities(idxs, td_err)

                self.learn_step += 1
                if self.learn_step % self.target_update == 0:
                    self.target_net.load_state_dict(self.q_net.state_dict())

        # return metrics for this episode
        return total_reward, length, inad / length

    def save(self, ckpt_dir, tag, seed, ep=None):
        os.makedirs(ckpt_dir, exist_ok=True)
        if ep:
            pfn = f"{tag}_seed{seed}_ep{ep}_q.pth"
            tfn = f"{tag}_seed{seed}_ep{ep}_target.pth"
        else:
            pfn = f"{tag}_seed{seed}_final_q.pth"
            tfn = f"{tag}_seed{seed}_final_target.pth"
        torch.save(self.q_net.state_dict(),     os.path.join(ckpt_dir, pfn))
        torch.save(self.target_net.state_dict(),os.path.join(ckpt_dir, tfn))

    def load(self, ckpt_dir, tag, seed, ep):
        qf = os.path.join(ckpt_dir, f"{tag}_seed{seed}_ep{ep}_q.pth")
        tf = os.path.join(ckpt_dir, f"{tag}_seed{seed}_ep{ep}_target.pth")
        self.q_net.load_state_dict(torch.load(qf, map_location=self.device))
        self.target_net.load_state_dict(torch.load(tf, map_location=self.device))



In [ ]:
# reproducibility
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

env_name = 'Sepsis/ICU-Sepsis-v2'
agent = DQNAgent(
    env_name=env_name,
    lr=1e-3,
    gamma=0.99,
    buffer_size=10_000,
    batch_size=64,
    eps_start=1.0,
    eps_end=0.001,
    alpha=0.6,
    beta=0.4,
    learning_starts=10_000,
    train_freq=10,
    target_update=512,
    device='cpu'
)

N_EPISODES = 300_000
returns, lengths, inad_freqs = [], [], []

for ep in tqdm(range(1, N_EPISODES + 1), desc='DQN Test', unit='ep'):
    r, l, inad = agent.train_one_episode(ep, N_EPISODES)
    returns.append(r)
    lengths.append(l)
    inad_freqs.append(inad)

In [ ]:
# Smoothing helper
def smooth(x, w=100):
    return np.convolve(x, np.ones(w)/w, mode='same')

sr = smooth(np.array(returns))
sl = smooth(np.array(lengths))
si = smooth(np.array(inad_freqs))
cum_steps = np.cumsum(lengths)

# 1. Survival Rate
plt.figure()
plt.plot(sr)
plt.title('Survival Rate (Single Seed)')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.grid(True)
plt.show()

# 2. Episode Length
plt.figure()
plt.plot(sl)
plt.title('Episode Length (Single Seed)')
plt.xlabel('Episode')
plt.ylabel('Length (steps)')
plt.grid(True)
plt.show()

# 3. Inadmissible-Action Frequency
plt.figure()
plt.plot(si)
plt.title('Inadmissible-Action Frequency (Single Seed)')
plt.xlabel('Episode')
plt.ylabel('Inadmissible Action %')
plt.grid(True)
plt.show()

# 4. Sample Efficiency (Return vs Steps)
plt.figure()
plt.plot(cum_steps, sr)
plt.title('Sample Efficiency (Single Seed)')
plt.xlabel('Cumulative Env Steps')
plt.ylabel('Return')
plt.grid(True)
plt.show()

In [ ]:
base_dir = '/Users/nicksmits/Library/CloudStorage/OneDrive-Personal(2)/COMP 579/Priority_DQN_Fast'
seeds = range(10)

# fixed params
batch_size = 128

# hyperparams to sweep
learning_rates = [1e-3, 5e-4]
gammas         = [0.99, 0.995]

episodes = 300_000
checkpoint_interval = 100_000

for lr, gamma in product(learning_rates, gammas):
    tag = f"DQN_lr{lr}_bs{batch_size}_gamma{gamma}"
    ckpt_dir = os.path.join(base_dir, tag)

    for seed in seeds:
        # reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        # metric file paths
        f_ret  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_returns.npy")
        f_len  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_lengths.npy")
        f_inad = os.path.join(ckpt_dir, f"{tag}_seed{seed}_inad.npy")

        # resume existing metrics
        returns = list(np.load(f_ret))  if os.path.exists(f_ret)  else []
        lengths = list(np.load(f_len))  if os.path.exists(f_len)  else []
        inads   = list(np.load(f_inad)) if os.path.exists(f_inad) else []

        # determine where to resume
        start_ep = len(returns) + 1
        max_ep = 0
        if os.path.isdir(ckpt_dir):
            pat = re.compile(fr"{re.escape(tag)}_seed{seed}_ep(\d+)_q\.pth")
            for fn in os.listdir(ckpt_dir):
                m = pat.match(fn)
                if m:
                    epn = int(m.group(1))
                    max_ep = max(max_ep, epn)
            if max_ep:
                start_ep = max_ep + 1

        # instantiate agent (no eps_decay arg)
        agent = DQNAgent(
            env_name        = 'Sepsis/ICU-Sepsis-v2',
            lr              = lr,
            gamma           = gamma,
            buffer_size     = 10_000,
            batch_size      = batch_size,
            eps_start       = 1.0,
            eps_end         = 0.001,
            alpha           = 0.6,
            beta            = 0.4,
            learning_starts = 10_000,
            train_freq      = 10,
            target_update   = 5_000,
            device          = 'cpu'
        )

        # if resuming from checkpoint, load networks
        if max_ep:
            agent.load(ckpt_dir, tag, seed, max_ep)

        # training loop
        pbar = tqdm(range(start_ep, episodes+1),
                    desc=f"{tag} | seed{seed}", unit='ep')
        for ep in pbar:
            r, length, inad_freq = agent.train_one_episode(ep, episodes)
            returns.append(r)
            lengths.append(length)
            inads.append(inad_freq)
            pbar.set_postfix({"r":f"{r:.2f}", "len":length, "inad%":f"{inad_freq:.2%}"})

            # checkpoint & save metrics
            if ep % checkpoint_interval == 0:
                agent.save(ckpt_dir, tag, seed, ep)
                np.save(f_ret,  np.array(returns))
                np.save(f_len,  np.array(lengths))
                np.save(f_inad, np.array(inads))

        # final save after all episodes
        agent.save(ckpt_dir, tag, seed)
        np.save(f_ret,  np.array(returns))
        np.save(f_len,  np.array(lengths))
        np.save(f_inad, np.array(inads))

In [ ]:
from pathlib import Path
# Configuration (reuse from before)
base_dir = "/Users/nicksmits/Library/CloudStorage/OneDrive-Personal(2)/COMP 579/Priority_DQN_Fast"
base = Path(base_dir)
seeds = range(5)

learning_rates = [1e-3, 5e-4]
gammas         = [0.99, 0.995]

window = 5000

def smooth(x, w=window):
    return np.convolve(x, np.ones(w)/w, mode='same')


# Gather all tags
tags = []
for lr in learning_rates:
    for gamma in gammas:
        tags.append(f"DQN_lr{lr}_bs64_gamma{gamma}")

# for tag in tags:
#     ckpt_dir = base / tag
#     print(f"\nLooking in: {ckpt_dir}")
#     if not ckpt_dir.exists():
#         print("  → directory does NOT exist!")
#         continue

#     # list everything in it
#     all_files = sorted([p.name for p in ckpt_dir.iterdir()])
#     print("  contains:", all_files)

#     # now look for your *.npy
#     pattern = str(ckpt_dir / f"{tag}_seed*_returns.npy")
#     matches = glob.glob(pattern)
#     print("  matched by glob:", [Path(m).name for m in matches])

# Prepare plot
plt.figure(figsize=(10,6))
colors = plt.cm.tab10(np.linspace(0,1,len(tags)))

for color, tag in zip(colors, tags):
    ckpt_dir = os.path.join(base_dir, tag)
    # Load returns for all seeds
    rets = []
    for seed in seeds:
        f_ret = os.path.join(ckpt_dir, f"{tag}_seed{seed}_returns.npy")
        print(f_ret)
        if os.path.exists(f_ret):
            rets.append(np.load(f_ret))
    if not rets:
        continue

    rets = np.stack(rets)          # shape (n_seeds, episodes)
    mean_r = rets.mean(axis=0)     # [episodes]
    std_r  = rets.std(axis=0)

    # smooth mean only
    sr = smooth(mean_r)
    std_r = smooth(std_r)
    sstd = std_r / np.sqrt(3)
    # clamp shading bounds [0,1]
    lower = np.maximum(sr - sstd, 0.0)
    upper = np.minimum(sr + sstd, 1.0)

    episodes = len(sr)
    x = np.arange(episodes)

    # plot
    plt.plot(x, sr, color=color, label=tag)
    plt.fill_between(x, lower, upper, color=color, alpha=0.15)

plt.title("Survival Rate Curves Across DQN Hyperparameters")
plt.xlabel("Episode")
plt.ylabel("Survival Rate (Smoothed)")
plt.ylim(0.74,0.86)
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show()

In [ ]:
# Configuration (reuse from before)
base_dir = "/Users/nicksmits/Library/CloudStorage/OneDrive-Personal(2)/COMP 579/Priority_DQN_Fast"
base = Path(base_dir)
seeds = range(3)

learning_rates = [1e-3, 5e-4]
gammas         = [0.99, 0.995]

window = 10000

def smooth(x, w=window):
    return np.convolve(x, np.ones(w)/w, mode='same')

# Gather all tags
tags = []
for lr in learning_rates:
    for gamma in gammas:
        tags.append(f"DQN_lr{lr}_bs64_gamma{gamma}")

# for tag in tags:
#     ckpt_dir = base / tag
#     print(f"\nLooking in: {ckpt_dir}")
#     if not ckpt_dir.exists():
#         print("  → directory does NOT exist!")
#         continue

#     # list everything in it
#     all_files = sorted([p.name for p in ckpt_dir.iterdir()])
#     print("  contains:", all_files)

#     # now look for your *.npy
#     pattern = str(ckpt_dir / f"{tag}_seed*_returns.npy")
#     matches = glob.glob(pattern)
#     print("  matched by glob:", [Path(m).name for m in matches])

# Prepare plot
plt.figure(figsize=(10,6))
# colors = plt.cm.tab10(np.linspace(0,1,len(tags)))
color = plt.colormaps.get_cmap('tab10')

for color, tag in zip(colors, tags):
    ckpt_dir = os.path.join(base_dir, tag)
    # Load returns for all seeds
    lens = []
    for seed in seeds:
        f_len = os.path.join(ckpt_dir, f"{tag}_seed{seed}_lengths.npy")
        print(f_len)
        if os.path.exists(f_ret):
            lens.append(np.load(f_len))
    if not lens:
        continue

    lens = np.stack(lens)          # shape (n_seeds, episodes)
    mean_l = lens.mean(axis=0)     # [episodes]
    std_l  = lens.std(axis=0)

    # smooth mean only
    sl = smooth(mean_l)
    std_l = smooth(std_l)
    sstd = std_l / np.sqrt(len(seeds))

    episodes = len(sl)
    x = np.arange(episodes)

    lower = sl - sstd
    upper = sl + sstd

    # plot
    plt.plot(x, sl, color=color, label=tag)
    plt.fill_between(x, lower, upper, color=color, alpha=0.15)

plt.title("Episode Length Curves Across DQN Hyperparameters")
plt.xlabel("Episode")
plt.ylabel("Episode Length (Smoothed)")
plt.ylim(9, 13)
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show()